In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os, sys

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -----------------------------
# Add shared utilities path
# -----------------------------
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")


from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema, get_features_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
gold_schema = get_gold_schema()    # e.g., dev_gold
features_schema = get_features_schema() 

from superstore_logger import get_superstore_logger, log_event
from superstore_backfill_utils import is_dry_run
# -----------------------------
# log_event Setup 
# -----------------------------
# Initialize log_event to capture events 
logger_features_customers = get_superstore_logger("superstore_features_customers")


# -----------------------------
# Dry run
# -----------------------------
# The serving layer rebuilds from Gold and has no run-mode logic of its own —
# a replay or backfill simply produces different Gold for it to read. But it
# does write, so it must not write during a dry run. is_dry_run reads only the
# dry_run parameter and never raises, so a serving task cannot fail on the
# full_refresh gate that is enforced upstream where it belongs.
if is_dry_run(dbutils):
    log_event(
        logger_features_customers,
        "INFO",
        "Dry run: skipping features_customer write",
        dry_run=True,
        layer="Feature Engineering"
    )
    dbutils.notebook.exit("DRY_RUN_COMPLETED layer=features_customer (no data written)")

# -------------------------------
# Function to check if the table exists
# -------------------------------
def check_table_exists(catalog, schema, table_name):
    try:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        return spark.catalog.tableExists(full_table_name)
    except Exception as e:
        log_event(
            logger_features_customers,
            "ERROR",
            f"Error checking table existence: {e}",
            layer="Feature Engineering"
        )
        return False

# -------------------------------
# Required table to create or update features Table (Environment-aware)
# -------------------------------
required_tables = [
    f"{catalog}.{gold_schema}.dim_customers",
    f"{catalog}.{gold_schema}.facts_orders",
    f"{catalog}.{gold_schema}.facts_sales"
]

missing_tables = []

# Check each table
for table in required_tables:
    if not check_table_exists(catalog, gold_schema, table.split('.')[-1]):
        missing_tables.append(table)

if missing_tables:
    log_event(logger_features_customers, "ERROR", f"Missing tables: {', '.join(missing_tables)}. Cannot create feature table.", layer="Feature Engineering")
else:
    # -------------------------------
    # Create or Update Customer Features Table (Environment-aware)
    # -------------------------------
    if spark.catalog.tableExists(f"{catalog}.{features_schema}.features_customer"):
        # If the table exists, perform a MERGE operation to update or insert new data
        spark.sql(f"""
            MERGE INTO {catalog}.{features_schema}.features_customer AS target
            USING (
                WITH customer_orders AS (
                    -- Aggregating order details like sales, profit, and discount for each customer
                    SELECT
                        c.customer_id,
                        o.order_id,
                        o.order_date,
                        sum(s.sales) AS total_sales,
                        sum(s.profit) AS total_profit,
                        avg(s.discount) AS avg_discount,
                        c.region,
                        c.segment
                    FROM {catalog}.{gold_schema}.dim_customers c
                    JOIN {catalog}.{gold_schema}.facts_orders o ON c.customer_id = o.customer_id
                    JOIN {catalog}.{gold_schema}.facts_sales s ON o.order_id = s.order_id
                    GROUP BY c.customer_id, o.order_id, o.order_date, c.region, c.segment
                ),
                customer_recency AS (
                    -- Compute recency as the number of days since the last order for each customer
                    SELECT
                        customer_id,
                        CURRENT_DATE() AS feature_date,
                        DATEDIFF(CURRENT_DATE(), MAX(order_date)) AS recency_days
                    FROM customer_orders
                    GROUP BY customer_id
                ),
                customer_frequency AS (
                    -- Compute frequency of orders for the last 30 and 90 days
                    SELECT
                        customer_id,
                        SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 30) THEN 1 ELSE 0 END) AS order_frequency_30days,
                        SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 90) THEN 1 ELSE 0 END) AS order_frequency_90days
                    FROM customer_orders
                    GROUP BY customer_id
                ),
                customer_monetary AS (
                    -- Compute monetary value in the last 30 and 90 days, along with average discount
                    SELECT
                        customer_id,
                        SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 30) THEN total_sales ELSE 0 END) AS monetary_30days,
                        SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 90) THEN total_sales ELSE 0 END) AS monetary_90days,
                        AVG(avg_discount) AS avg_discount
                    FROM customer_orders
                    GROUP BY customer_id
                ),
                customer_profit_variance AS (
                    -- Compute profit margin and order variance for each customer
                    SELECT
                        customer_id,
                        CASE 
                            WHEN SUM(total_sales) = 0 THEN 0
                            ELSE ROUND(SUM(total_profit)/SUM(total_sales), 4)
                        END AS profit_margin,
                        CASE 
                            WHEN COUNT(order_id) > 1 THEN ROUND(VAR_SAMP(total_sales), 2)
                            ELSE 0
                        END AS order_variance
                    FROM customer_orders
                    GROUP BY customer_id
                ),
                customer_encoding AS (
                    -- Encode segment and region using DENSE_RANK() to map categorical values to integers
                    SELECT
                        customer_id,
                        DENSE_RANK() OVER (ORDER BY segment) AS segment_encoded,
                        DENSE_RANK() OVER (ORDER BY region) AS region_encoded
                    FROM customer_orders
                    GROUP BY customer_id, segment, region
                ),
                -- Apply ROW_NUMBER to eliminate duplicate rows for the same customer_id and feature_date
                ranked_data AS (
                    SELECT 
                        r.customer_id,
                        r.feature_date,
                        r.recency_days,
                        f.order_frequency_30days,
                        f.order_frequency_90days,
                        m.monetary_30days,
                        m.monetary_90days,
                        m.avg_discount,
                        p.profit_margin,
                        p.order_variance,
                        e.segment_encoded,
                        e.region_encoded,
                        ROW_NUMBER() OVER (PARTITION BY r.customer_id, r.feature_date ORDER BY r.feature_date DESC) AS row_num
                    FROM customer_recency r
                    JOIN customer_frequency f ON r.customer_id = f.customer_id
                    JOIN customer_monetary m ON r.customer_id = m.customer_id
                    JOIN customer_profit_variance p ON r.customer_id = p.customer_id
                    JOIN customer_encoding e ON r.customer_id = e.customer_id
                )
                -- Only keep the first row for each customer_id and feature_date pair
                SELECT * FROM ranked_data WHERE row_num = 1
            ) AS source
            ON target.customer_id = source.customer_id
            AND target.feature_date = source.feature_date
            WHEN MATCHED THEN
                UPDATE SET
                    target.recency_days = source.recency_days,
                    target.order_frequency_30days = source.order_frequency_30days,
                    target.order_frequency_90days = source.order_frequency_90days,
                    target.monetary_30days = source.monetary_30days,
                    target.monetary_90days = source.monetary_90days,
                    target.avg_discount = source.avg_discount,
                    target.profit_margin = source.profit_margin,
                    target.order_variance = source.order_variance,
                    target.segment_encoded = source.segment_encoded,
                    target.region_encoded = source.region_encoded
            WHEN NOT MATCHED THEN
                INSERT (customer_id, feature_date, recency_days, order_frequency_30days, order_frequency_90days,
                        monetary_30days, monetary_90days, avg_discount, profit_margin, order_variance,
                        segment_encoded, region_encoded)
                VALUES (source.customer_id, source.feature_date, source.recency_days, source.order_frequency_30days,
                        source.order_frequency_90days, source.monetary_30days, source.monetary_90days,
                        source.avg_discount, source.profit_margin, source.order_variance, source.segment_encoded,
                        source.region_encoded)
        """)

    else:
        # Table doesn't exist, create and insert data
        spark.sql(f"""
            -- Create the Customer ML Features Table if it doesn't exist
            CREATE TABLE {catalog}.{features_schema}.features_customer AS
            -- Repeatedly include the same logic used in the MERGE block to create the table from scratch
            WITH customer_orders AS (
                SELECT
                    c.customer_id,
                    o.order_id,
                    o.order_date,
                    sum(s.sales) AS total_sales,
                    sum(s.profit) AS total_profit,
                    avg(s.discount) AS avg_discount,
                    c.region,
                    c.segment
                FROM {catalog}.{gold_schema}.dim_customers c
                JOIN {catalog}.{gold_schema}.facts_orders o ON c.customer_id = o.customer_id
                JOIN {catalog}.{gold_schema}.facts_sales s ON o.order_id = s.order_id
                GROUP BY c.customer_id, o.order_id, o.order_date, c.region, c.segment
            ),
            customer_recency AS (
                SELECT
                    customer_id,
                    CURRENT_DATE() AS feature_date,
                    DATEDIFF(CURRENT_DATE(), MAX(order_date)) AS recency_days
                FROM customer_orders
                GROUP BY customer_id
            ),
            customer_frequency AS (
                SELECT
                    customer_id,
                    SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 30) THEN 1 ELSE 0 END) AS order_frequency_30days,
                    SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 90) THEN 1 ELSE 0 END) AS order_frequency_90days
                FROM customer_orders
                GROUP BY customer_id
            ),
            customer_monetary AS (
                SELECT
                    customer_id,
                    SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 30) THEN total_sales ELSE 0 END) AS monetary_30days,
                    SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 90) THEN total_sales ELSE 0 END) AS monetary_90days,
                    AVG(avg_discount) AS avg_discount
                FROM customer_orders
                GROUP BY customer_id
            ),
            customer_profit_variance AS (
                SELECT
                    customer_id,
                    CASE 
                        WHEN SUM(total_sales) = 0 THEN 0
                        ELSE ROUND(SUM(total_profit) / SUM(total_sales), 4)
                    END AS profit_margin,
                    CASE 
                        WHEN COUNT(order_id) > 1 THEN ROUND(VAR_SAMP(total_sales), 2)
                        ELSE 0
                    END AS order_variance
                FROM customer_orders
                GROUP BY customer_id
            ),
            customer_encoding AS (
                SELECT
                    customer_id,
                    DENSE_RANK() OVER (ORDER BY segment) AS segment_encoded,
                    DENSE_RANK() OVER (ORDER BY region) AS region_encoded
                FROM customer_orders
                GROUP BY customer_id, segment, region
            )
            SELECT
                r.customer_id,
                r.feature_date,
                r.recency_days,
                f.order_frequency_30days,
                f.order_frequency_90days,
                m.monetary_30days,
                m.monetary_90days,
                m.avg_discount,
                p.profit_margin,
                p.order_variance,
                e.segment_encoded,
                e.region_encoded
            FROM customer_recency r
            JOIN customer_frequency f ON r.customer_id = f.customer_id
            JOIN customer_monetary m ON r.customer_id = m.customer_id
            JOIN customer_profit_variance p ON r.customer_id = p.customer_id
            JOIN customer_encoding e ON r.customer_id = e.customer_id
        """)
log_event(logger_features_customers, "INFO", "Feature customer table created successfully.", layer="Feature Engineering")
